# Task 2: End-to-End ML Pipeline with Scikit-learn
## Predicting Customer Churn

**Objective:** Build a reusable, production-ready ML pipeline that predicts whether a customer will churn (leave the service).

**Dataset:** Telco Customer Churn  
**Models:** Logistic Regression, Random Forest  
**Key Tools:** scikit-learn Pipeline, GridSearchCV, joblib


## Step 1: Importign All Required Libraries

In [1]:
# Saari libraries aik saath hi import kr liii 

import pandas as pd
import joblib

from sklearn.model_selection import GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
from sklearn.metrics import f1_score


## 2. Load the Dataset


In [2]:
df = pd.read_csv('WA_Fn-UseC_-Telco-Customer-Churn.csv')
df.head()

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


## 3. Data Cleaning & Preprocessing

In [3]:
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
df = df.drop(columns=['customerID']) # no use actually

df['Churn'] = df['Churn'].map({
    'Yes': 1,
    'No': 0
})

print(df['Churn'].value_counts())

Churn
0    5174
1    1869
Name: count, dtype: int64


## Step 4: Separate Features and Target, Then Split into Train/Test

In [4]:
X = df.drop(columns=['Churn']) # feature except churn (i/p features)
y = df['Churn'] # target column (o/p feature)

print('Input features table shape:', X.shape)
print('Target column shape:', y.shape)

Input features table shape: (7043, 19)
Target column shape: (7043,)


In [5]:
numeric_columns = []
categorical_columns = []

for col in X.columns:
    if X[col].dtype == 'object':
        categorical_columns.append(col)
    else:
        numeric_columns.append(col)

print('Numeric columns:')
print(numeric_columns)

print('\nCategorical columns:')
print(categorical_columns)

# piepeline me use honge ye aagey

Numeric columns:
['SeniorCitizen', 'tenure', 'MonthlyCharges', 'TotalCharges']

Categorical columns:
['gender', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod']


In [6]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

## 5. Building the Scikit-learn Pipeline


In [7]:
# creating pipeline for numeric data first
numeric_pipeline = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler',  StandardScaler())
])

# creating pipeline for categorical data
categorical_pipeline = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore'))
])

print('Numeric pipeline done.')
print('Categorical pipeline done.')

Numeric pipeline done.
Categorical pipeline done.


In [8]:
# Combine both pipelines using ColumnTransformer
preprocessor = ColumnTransformer(transformers=[
    ('numeric',      numeric_pipeline,      numeric_columns),
    ('categorical',  categorical_pipeline,  categorical_columns)
])

print('Preprocessor (ColumnTransformer) built successfully.')

Preprocessor (ColumnTransformer) built successfully.


## Step 6: Attach Models to the Preprocessor (Full Pipelines)

In [9]:
# Full pipeline 1: Preprocessor + Logistic Regression

lr_pipeline = Pipeline(steps=[
    ('preprocessor',       preprocessor),
    ('logistic_regression', LogisticRegression(max_iter=1000, random_state=42))
])

# Full pipeline 2: Preprocessor + Random Forest

rf_pipeline = Pipeline(steps=[
    ('preprocessor',    preprocessor),
    ('random_forest',   RandomForestClassifier(n_estimators=100, random_state=42))
])

print('Both full pipelines created.')

Both full pipelines created.


## Step 7: Training Both Models

In [10]:
# Training Logistic Regression
print('Training Logistic Regression:')
lr_pipeline.fit(X_train, y_train)
print('Logistic Regression training done.')

lr_predictions = lr_pipeline.predict(X_test) # predictions

# Calculate metrics (just for my own understanding; not necessary)
lr_accuracy = accuracy_score(y_test, lr_predictions)
lr_f1       = f1_score(y_test, lr_predictions)

print('Logistic Regression Results')
print('Accuracy:', round(lr_accuracy * 100, 2), '%')
print('F1 Score:', round(lr_f1, 4))

Training Logistic Regression:
Logistic Regression training done.
Logistic Regression Results
Accuracy: 82.11 %
F1 Score: 0.64


In [11]:
# Train Random Forest
print('Training Random Forest...')
rf_pipeline.fit(X_train, y_train)
print('Training Random Forest done.')

rf_predictions = rf_pipeline.predict(X_test) # doing predictions

# Calculate metrics (again: not mandatory, just for my understanding)
rf_accuracy = accuracy_score(y_test, rf_predictions)
rf_f1       = f1_score(y_test, rf_predictions)

print('Random Forest Results')
print('Accuracy:', round(rf_accuracy * 100, 2), '%')
print('F1 Score:', round(rf_f1, 4))

Training Random Forest...
Training Random Forest done.
Random Forest Results
Accuracy: 79.42 %
F1 Score: 0.5469


## Step 8: Hyperparameter Tuning with GridSearchCV

We'll tune the **Random Forest** since it has more tunable parameters.


In [12]:
# in a Pipeline, parameter names follow the format:
#   'step_name__parameter_name' 
param_grid = {
    'random_forest__n_estimators':  [100, 200],
    'random_forest__max_depth':      [None, 10, 20],
    'random_forest__min_samples_split': [2, 5]
}

In [13]:
grid_search = GridSearchCV(
    estimator=rf_pipeline,
    param_grid=param_grid,
    cv=5,
    scoring='f1',
    n_jobs=-1,
    verbose=2
)

grid_search.fit(X_train, y_train)
print('GridSearchCV complete!')

Fitting 5 folds for each of 12 candidates, totalling 60 fits
GridSearchCV complete!


In [14]:
print('Best Parameters Found:', grid_search.best_params_)

print('Best Cross-Validation F1 Score:', round(grid_search.best_score_, 4))

Best Parameters Found: {'random_forest__max_depth': 10, 'random_forest__min_samples_split': 5, 'random_forest__n_estimators': 100}
Best Cross-Validation F1 Score: 0.5683


## Step 9: Export the Pipeline with joblib

In [15]:
# Saving the complete best pipeline to a file
best_model = grid_search.best_estimator_
best_predictions = best_model.predict(X_test)

joblib.dump(best_model, 'churn_pipeline.pkl')

print('Pipeline saved as churn_pipeline.pkl')

Pipeline saved as churn_pipeline.pkl


In [16]:
# just a little verification

loaded_pipeline = joblib.load('churn_pipeline.pkl')
loaded_predictions = loaded_pipeline.predict(X_test)

all_same = (loaded_predictions == best_predictions).all()

print('Loaded pipeline predictions match original:', all_same)
print('Export verified successfully!')

Loaded pipeline predictions match original: True
Export verified successfully!
